# IRWA Final Project — Part 2: Indexing and Evaluation

**Educational, reproducible working notebook (2026).** Uses the professor's fashion-products dataset and builds on the original Part 1 preprocessing decisions. The original team's notebooks have been preserved in the repository.

**Goal:** Build an inverted index, implement Boolean AND search, rank results with TF-IDF, then evaluate with the required metrics. **Human judgments for five group-defined queries must be completed by the group.**

> Setup: Open the **repository root** in VS Code, install `requirements.txt`, and run `python -m nltk.downloader stopwords` once. The notebook uses the project modules so the code and the report can be reproduced without copying formulas into multiple places.

## 1. Load and preprocess the data (Part 1 dependency)

Part 1's original notebook normalizes the fields `title`, `description`, `brand`, `category`, `sub_category`, `seller`, and values from `product_details`. It preserves `pid` and structured price/rating/stock metadata.

The helper `prepare_data()` recreates the two output CSVs without running Part 1's optional charts or spaCy NER. If your group already has `data/fashion_products.csv` and `data/index_info.csv`, the main code reuses them.

In [ ]:
from pathlib import Path
import pandas as pd
from project_progress.part_2.prepare_data import prepare_data, DATA
from project_progress.part_2.search_engine import (
    load_processed, build_inverted_index, calculate_idf,
    normalize_tf, rank_documents, conjunctive_search, preprocess
)
from project_progress.part_2.evaluation import evaluate_qrels, evaluate, aggregate
from project_progress.part_2.run_part2 import CUSTOM, VALIDATION, RESULTS

if not (DATA / "index_info.csv").exists():
    prepare_data(DATA)
products, index_df = load_processed(DATA)
print(f"Products: {len(products):,}")
print(f"Fields: {list(index_df.columns)}")
index_df.head(3)

## 2. Build a field-aware positional inverted index

A term is indexed as `term -> field -> product-ID -> positions`. Every field keeps its own positions, and a term can appear in multiple fields of the same product.

For **AND retrieval**, all non-stopword terms must occur somewhere in the product; term occurrence across different fields still counts. Document frequency counts each product once per term. Term frequency counts repeated occurrences across all seven fields.

In [ ]:
inverted_index, tf, df = build_inverted_index(index_df)
idf = calculate_idf(df, len(index_df))
normalized_tf = normalize_tf(tf)
print(f"Vocabulary size: {len(inverted_index):,}")
print("Example: normalized query", preprocess("Women cotton sweatshirt"))
print("AND matches:", len(conjunctive_search("women cotton sweatshirt", inverted_index)))
print("Example posting fields for cotton:", list(inverted_index.get('cotton', {}).keys()))

## 3. Boolean AND + TF-IDF ranking

We use the group's original scoring convention:

- `idf(t) = ln(N/df(t))`
- `TF_doc(t) = count(t, d) / sqrt(sum_all_terms(count(term, d)^2))`
- `TF_query(t) = count(t, q) / sqrt(sum_query_terms(count(term,q)^2))`
- `score(q,d) = sum_over_query_terms [TF_doc * IDF * TF_query * IDF]`

This is a TF-IDF weighted **dot product**, not cosine similarity of final normalized TF-IDF vectors. AND-filtered products are sorted by descending score, with PID used to break ties reproducibly.

In [ ]:
for num, query in enumerate(CUSTOM, start=1):
    ranking = rank_documents(query, inverted_index, normalized_tf, idf)
    print(f"{num}. {query!r}: {len(ranking)} matching products")
    for pid, score in ranking[:3]:
        title = products.loc[products.pid == pid, 'title'].iloc[0]
        print(f"    {score:.4f}  {pid}  {title}")

## 4. Implement and test evaluation metrics

Metrics used: Precision@K, Recall@K, AP@K, F1@K, MAP, MRR and NDCG@K. These are implemented in `evaluation.py`.

The validation file contains **partial** relevance judgments (20 products for each predefined query). To avoid treating all unjudged products as nonrelevant, we rank on the full corpus, then **condense the ranking to judged products**, preserving their order. Relevant judged products not returned by AND still count toward the recall denominator. We report at `K=10` and use `min(K, known_relevant)` for the AP@K denominator.

In [ ]:
validation = pd.read_csv(DATA / "validation_labels.csv")
K = 10
per_query = []
rows = []
for qid, query in VALIDATION.items():
    labels = validation.loc[validation.query_id == qid]
    qrels = dict(zip(labels.pid, labels.labels))
    full_ranking = rank_documents(query, inverted_index, normalized_tf, idf)
    metrics, ranked_judged = evaluate_qrels(query, full_ranking, qrels, k=K)
    per_query.append(metrics)
    rows.append({"Query": qid, "P@10":metrics["Precision"], "R@10":metrics["Recall"],
                 "AP@10":metrics["AP"], "F1@10":metrics["F1"], "NDCG@10":metrics["NDCG"]})
results = pd.DataFrame(rows)
print(results.to_string(index=False, float_format=lambda x: f"{x:.3f}"))
print(pd.DataFrame([{"MAP@10":aggregate(per_query)["MAP"], "MRR@10":aggregate(per_query)["MRR"]}]).to_string(index=False, float_format=lambda x: f"{x:.3f}"))

## 5. Generate everything required for the report and human judgment

The next cell writes the same outputs to `results/` and exports an **unlabeled** CSV with every retrieved custom-query product (327 rows using the supplied data). The five-query ground truth **must be created by your group**. Do not treat suggested or automated labels as expert judgments.

In [ ]:
from project_progress.part_2.run_part2 import run
metadata, validation_table, global_metrics = run(k=10, prepare_missing=False)
print("Full human judgment sheet:", DATA / "human_judgments_ALL_BLANK.csv")
print("Optional top-10 sheet:", DATA / "human_judgments_TOP10_BLANK.csv")

## 6. When your group has completed every human label

Open the CSV, review each product against the **shopper's information need**, and fill `label` with 1 (relevant) or 0 (not relevant). Once saved, run the next cell. This produces metric tables for the five custom queries.

**Limitation:** If you only judge retrieved products, the custom-query recall denominator is restricted to that judged pool, not the entire fashion corpus. Explain that openly in the report.

In [ ]:
from project_progress.part_2.evaluate_human import evaluate_file
sheet = DATA / "human_judgments_ALL_BLANK.csv"
try:
    evaluate_file(sheet, k=10)
except ValueError as err:
    print("Human annotation not finished:", err)

## 7. Reproducibility and final delivery

- Run `python -m unittest discover -s tests -v` from the repository root.
- Use `results/validation_metrics.csv` and `results/validation_aggregate.csv` for the numerical-only instructor-query results section.
- Insert five-query human-judgment scores and interpret the differences between metrics in `report_draft.pdf` after reviewing results yourselves.
- Confirm names/IDs, GitHub URL, and tag; put the PDF on Aula Global and tag the exact committed version.
- Disclose AI coding and editing assistance accurately. Do not submit the draft report verbatim without the group's own analysis.